[Reference](https://medium.com/activated-thinker/build-a-complete-langchain-rag-app-for-free-e9b535f20595)

# Install All Required Libraries

In [1]:
!pip install langchain langchain-core langchain-community chromadb sentence-transformers pypdf transformers accelerate huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

# Login to Hugging Face (Gemma Requires Access)

In [3]:
from huggingface_hub import login
# This will prompt for your HF token
login()

# Load the Free Gemma LLM

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)

# Load a PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
docs = loader.load()

# Split PDF Into Chunks

In [6]:
from langchain_text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

In [7]:
chunks = splitter.split_documents(docs)

# Create Free Embeddings

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create Vector Store (Chroma)

In [9]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(chunks, embed)
retriever = db.as_retriever()

# Create a Custom LLM Wrapper for Gemma (Important!)

In [10]:
from langchain_core.language_models import LLM
from langchain_core.outputs import Generation, LLMResult

class LocalLLM(LLM):
    def _call(self, prompt: str, stop=None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.2
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    @property
    def _identifying_params(self):
        return {"name": "gemma-local"}
    @property
    def _llm_type(self):
        return "custom"

# Build the RAG Chain Using LangChain Classic

In [11]:
from langchain_classic.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)